In [ ]:
import os

import flatdict as fd
import numpy as np
import pandas as pd
import requests
from odf.opendocument import load

from pynxtools_apm.examples.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
config: dict[str, str] = {
    "python_version": f"{sys.version.replace(' ', '_')}",
    "working_directory": f"{os.getcwd()}",
    "pynxtools_apm version": f"{get_pynxtools_apm_version()}",
    "rarfile version": f"{rarfile.__version__}",
    "sevenzip version": f"{py7zr.__version__}",
    # "blake3-py version": f"{blake3.__version__}",
    # "blake3-py max_threads": f"{blake3.blake3.AUTO}",
    # "directory": f"src_directory,  # sys.argv[1],
}

## Assure that we have for each dataset copyright information and assure that it matches with those from OpenAlex

In [ ]:
# https://spdx.org/licenses
# https://developers.openalex.org
# build a look-up table of bib keys
import webbrowser
from json import JSONDecodeError

for row in spread_sheet_of_all_projects.itertuples(index=True):
    # if row.parse in (1, 2):  # get cross-ref data if available if dataset is CC0 or CC-BY-4.0
    if int(row.project_name) < 1:  # skip all before
        continue
    if int(row.project_name) > 822:  # skip all after
        continue
    if row.license != "":
        continue
    project_id = get_project_id(f"{row.project_name}")
    match = []
    token = f"D{project_id}"
    for key in bib:
        if not key.startswith(token):
            continue
        if key not in match:
            match = [key]
        else:
            print(f"ERROR::Multiple matching D{project_id}* keys")
    if len(match) != 1:
        pass
        # print(f"ERROR::D{project_id}* not found")
    else:
        # print(f"{project_id}, {match[0]}")
        if "doi" in bib[match[0]]:
            doi = bib[match[0]]["doi"]
            webbrowser.open_new_tab(f"https://doi.org/{doi}")
            continue
            if doi.startswith("10.5281"):
                url = f"https://zenodo.org/api/records/{doi.replace('10.5281/zenodo.', '')}"
                try:
                    data = requests.get(url, timeout=30).json()
                    print(
                        f"{project_id}, {match[0]}, {doi}, {data['metadata']['license']['id']}"
                    )
                    del data
                except (JSONDecodeError, KeyError):
                    pass
                # time.sleep(1.0)
                # Zenodo anonymous API access rate limit 60/min, currently code though is slower than 1s per project
                del url
        else:
            pass
            # print(f"{project_id}, no DOI")
    del project_id, match, token
    continue

    if row.license not in ("CC BY 4.0", "CC0 1.0", "CC BY 4.0 / CC0 1.0"):
        pass
        # search matching reference from bib, take advantage that each reference always either startswith
        # f"D{project_id}" (dataset) or f"A{project_id}" (associated key paper)

In [ ]:
json_files = set()
for root, dirs, files in os.walk("openalex"):
    for file in files:
        # fpath = f"{root}/{file}".replace(os.sep * 2, os.sep)
        json_files.add(file)
print(len(json_files))

In [ ]:
mdata = fd.FlatDict(data, delimiter="/")
for key, obj in mdata.items():
    print(f"{key}, {obj}")

In [ ]:
authorships = data.get("authorships", [])
if len(authorships) == 0:
    print("No authorships")

first_author = authorships[0]

institutions = first_author.get("institutions", [])
if not institutions:
    print("No institutions")

first_institution = institutions[0]

result = {
    "doi": doi,
    "first_author": first_author.get("author", {}).get("display_name"),
    "institution": first_institution.get("display_name"),
    "country_code_iso3": first_institution.get("country_code"),
    "town": first_institution.get("city"),
}
print(result)

### Reformat the ods as that had successively grown becoming unmanagable, split these sheets up and standardize

* For each project sheet we do not want to have blank rows to concatenate the project sub-directory, sheet row_id for making entries and uploads be unique and descriptive.*
* sheets originally where all in one spreadsheet, which grew and become difficult to navigate, also not the typical use case, multiple people would puzzle together knowledge<br>
* Therefore, we disentangle here the original spreadsheet and clean it from blanks plus some more harmonization of formatting to reflect newer parsing capabilities for more<br>
  infrequently represented community file formats, typical NOMAD upload then aus_sydney_breen with entries named aus_sydney_breen_{1, 2, ...}

In [ ]:
run_sorting = False
if run_sorting:
    doc = load(f"{src_directory}{os.sep}harvest.examples.14.apm.xls.ods")

    spreadsheet = doc.spreadsheet
    sheets = [
        sheet
        for sheet in spreadsheet.childNodes
        if getattr(sheet, "tagName", "") == "table:table"
    ]
    sorted_sheets = sorted(sheets, key=lambda s: s.getAttribute("name"))
    for sheet in sheets:
        spreadsheet.removeChild(sheet)
    for sheet in sorted_sheets:
        spreadsheet.appendChild(sheet)

    doc.save(f"{src_directory}{os.sep}legacy_data_tmp.ods")

In [ ]:
# get sheet_names, modify these individually and export to project-specific file
# as the initial idea to have the sheets all together was simple when the collection
# was not that large but it becomes more and more impractical to navigate this
# large ODS sheet
doc = load(f"{src_directory}{os.sep}legacy_data_tmp.ods")
spreadsheet = doc.spreadsheet
sheet_names = [
    node.getAttribute("name")
    for node in spreadsheet.childNodes
    if getattr(node, "tagName", "") == "table:table"
]
# print(sheet_names)

Splitting it up makes also sense because legacy data mappings would typically be<br>
contributed efforts by multiple people. Which is what we are here mimicking.

In [ ]:
run_splitting = False
if run_splitting:
    column_header = [
        "str_rraw",
        "rhit_hits",
        "root",
        "pos_epos_apt_ato_csv",
        "rng_rrng_fig_env",
        "hdf_xml_nxs_raw_ops",
    ]
    for sheet_name in sheet_names[1:]:
        if sheet_name != "aaa_legacy_data":
            print(f"Validating {sheet_name}")
            df = pd.read_excel(
                f"{src_directory}{os.sep}legacy_data_tmp.ods",
                sheet_name=sheet_name,
                engine="odf",
            )
            df = df.fillna("")
            df = df[~((df.isna() | (df == "")).all(axis=1))]  # keep non-blank rows only

            # equalize number of columns
            if len(column_header) > df.shape[1]:  # needs adding empty columns
                for idx in range(0, len(column_header) - df.shape[1]):
                    df[f"new_column_{idx}"] = ""
                df.columns = column_header + [
                    f"ignore{col_idx}"
                    for col_idx in range(0, len(column_header) - df.shape[1])
                ]
            elif df.shape[1] > len(column_header):  # needs expanding the column_header
                df.columns = column_header + [
                    f"ignore{col_idx}"
                    for col_idx in range(0, df.shape[1] - len(column_header))
                ]
            else:
                df.columns = column_header + [
                    f"ignore{col_idx}"
                    for col_idx in range(0, df.shape[1] - len(column_header))
                ]

            # report issues with incorrectly categorized datasets
            for idx, row in df.iterrows():
                for col_name in df.columns:
                    if row[col_name] != "" and not col_name.startswith("ignore"):
                        if row[col_name][
                            row[col_name].rfind(".") + 1 :
                        ].lower() not in col_name.split("_"):
                            print(
                                f"WARNING, {sheet_name}, {idx}, {col_name}, {row[col_name]}"
                            )
                        # else:
                        #     print(f"{idx}, {col_name}, OK")

            df.to_excel(
                f"{src_directory}{os.sep}data{os.sep}{sheet_name}.ods",
                sheet_name=sheet_name,
                engine="odf",
                index=False,
            )
            del df

In [ ]:
issues = []
for line in issues:
    print(
        f"@Misc{{D_{''.join(token.capitalize() for token in line.split()[1].split('_'))},"
    )
    print(f"  note = {{personal communication with}},")
    print(f"}}")